In [ ]:
!pip install langgraph langchain langchain-groq wikipedia arxiv

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.8 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=1bfb9b46889e7b92fba378c1384af547c6bbac6e905c87e021ed0d4e2f9cd999
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [ ]:
import os
import arxiv
import wikipedia
from typing import TypedDict, List
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
# os.environ["GROQ_API_KEY"] = "gsk_..."

print("API key set!")

API key set!


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM ready!")

Groq LLM ready!


In [ ]:
def wikipedia_tool(topic: str) -> str:
    """Returns a short Wikipedia summary for a given topic."""
    try:
        # Use search first to find the best matching page title
        search_results = wikipedia.search(topic, results=3)
        if not search_results:
            return f"Wikipedia: No results found for '{topic}'."

        # Try the top search results until one works
        for title in search_results:
            try:
                summary = wikipedia.summary(title, sentences=5, auto_suggest=False)
                return f"[Wikipedia: {title}]\n{summary}"
            except wikipedia.exceptions.DisambiguationError as e:
                # Try first disambiguation option
                try:
                    summary = wikipedia.summary(e.options[0], sentences=5, auto_suggest=False)
                    return f"[Wikipedia: {e.options[0]}]\n{summary}"
                except Exception:
                    continue
            except Exception:
                continue

        return f"Wikipedia: Could not retrieve a summary for '{topic}'."
    except Exception as e:
        return f"Wikipedia search failed: {str(e)}"

# Quick test
print(wikipedia_tool("Transformer neural network"))

[Wikipedia: Transformer (deep learning)]
In deep learning, the transformer is a family of artificial neural network architectures based on the multi-head attention mechanism, in which text is converted to numerical representations called tokens, and each token is converted into a vector via lookup from a word embedding table. At each layer, each token is then contextualized within the scope of the context window with other (unmasked) tokens via a parallel multi-head attention mechanism, allowing the signal for key tokens to be amplified and less important tokens to be diminished. Because self-attention alone is permutation-invariant, transformers inject positional information, typically through positional encodings or learned positional embeddings, so token order can affect the output.
Transformers have the advantage of having no recurrent units, therefore requiring less training time than earlier recurrent neural architectures (RNNs) such as long short-term memory (LSTM). Later variat

In [ ]:
import arxiv
import time
from urllib.error import HTTPError


def arxiv_tool(research_topic: str) -> str:
    """
    Input:
        Research topic string

    Output:
        Top 3 Arxiv papers with abstracts
    """

    MAX_RETRIES = 5

    for attempt in range(MAX_RETRIES):

        try:
            # Safer client settings
            client = arxiv.Client(
                page_size=3,
                delay_seconds=5,
                num_retries=5
            )

            search = arxiv.Search(
                query=research_topic,
                max_results=3,
                sort_by=arxiv.SortCriterion.Relevance
            )

            results = list(client.results(search))

            if not results:
                return f"No Arxiv papers found for: {research_topic}"

            output = (
                f"\nTop 3 Arxiv Papers on "
                f"'{research_topic}'\n\n"
            )

            for i, paper in enumerate(results, start=1):

                # Clean abstract
                abstract = paper.summary.replace("\n", " ")

                # Truncate
                if len(abstract) > 400:
                    abstract = abstract[:400] + "..."

                # Authors
                authors = ", ".join(
                    author.name
                    for author in paper.authors[:3]
                )

                if len(paper.authors) > 3:
                    authors += " et al."

                output += (
                    f"Paper {i}: {paper.title}\n"
                    f"Authors: {authors}\n"
                    f"Published: "
                    f"{paper.published.strftime('%Y-%m-%d')}\n"
                    f"Abstract: {abstract}\n"
                    f"Link: {paper.entry_id}\n\n"
                )

            return output

        except HTTPError as e:

            # Handle rate limiting
            if e.code == 429:

                wait_time = 5 * (attempt + 1)

                print(
                    f"Rate limit hit. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                return f"HTTP Error: {str(e)}"

        except Exception as e:

            return f"Arxiv tool error: {str(e)}"

    return (
        "Arxiv API rate limit exceeded. "
        "Please try again later."
    )


# =========================
# TEST
# =========================

print(
    arxiv_tool("large language models")
)

Arxiv tool error: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=large+language+models&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=3)


In [ ]:
def llm_tool(query: str) -> str:
    """Answers a question directly using the LLM's built-in knowledge."""
    try:
        response = llm.invoke(query)
        return response.content
    except Exception as e:
        return f"LLM failed: {str(e)}"

# Quick test
print(llm_tool("What is the capital of Australia?"))

The capital of Australia is Canberra.


In [ ]:
class AgentState(TypedDict):
    question: str        # The user's question
    tool_used: str       # Which tool was selected by the router
    tool_output: str     # Raw output returned by the tool
    final_response: str  # Final polished answer for the user

print("AgentState defined!")

AgentState defined!


In [ ]:
def router_node(state: AgentState) -> AgentState:
    """Reads the question and picks the best tool."""
    question = state["question"]

    prompt = f"""You are a routing assistant. Read the question and decide which tool to use.

Available tools:
- wikipedia  : for concepts, definitions, history, general knowledge
- arxiv      : for latest research papers and academic topics
- llm        : for simple facts like capitals, dates, or general trivia
- out_of_scope: if the question is irrelevant, offensive, or cannot be answered

Question: {question}

Reply with ONLY one word from: wikipedia, arxiv, llm, out_of_scope"""

    response = llm.invoke(prompt)
    tool = response.content.strip().lower()

    # Make sure tool is valid, else default to llm
    valid_tools = ["wikipedia", "arxiv", "llm", "out_of_scope"]
    if tool not in valid_tools:
        tool = "llm"

    print(f"[Router] Tool selected: {tool}")
    return {**state, "tool_used": tool}

print("Router node defined!")

Router node defined!


In [ ]:
def tool_node(state: AgentState) -> AgentState:
    """Runs whichever tool the router selected."""
    question = state["question"]
    tool = state["tool_used"]

    if tool == "wikipedia":
        output = wikipedia_tool(question)
    elif tool == "arxiv":
        output = arxiv_tool(question)
    elif tool == "llm":
        output = llm_tool(question)
    elif tool == "out_of_scope":
        output = "Sorry, this question is out of scope for this research assistant."
    else:
        output = "Unknown tool. Could not process the question."

    return {**state, "tool_output": output}

print("Tool node defined!")

Tool node defined!


In [ ]:
def synthesiser_node(state: AgentState) -> AgentState:
    """Takes raw tool output and writes a clear final answer."""
    question = state["question"]
    tool_output = state["tool_output"]
    tool = state["tool_used"]

    # If out of scope or already a direct LLM answer, return as-is
    if tool in ["out_of_scope", "llm"]:
        return {**state, "final_response": tool_output}

    # For wikipedia and arxiv, ask LLM to clean up the answer
    prompt = f"""You are a research assistant. Using the information below, write a clear and structured answer to the question.

Question: {question}

Information gathered:
{tool_output}

Write a helpful, concise, well-structured response:"""

    try:
        response = llm.invoke(prompt)
        return {**state, "final_response": response.content}
    except Exception as e:
        return {**state, "final_response": tool_output}  # fallback to raw output

print("Synthesiser node defined!")

Synthesiser node defined!


In [ ]:
# Create the graph with AgentState
graph = StateGraph(AgentState)

# Add the three nodes
graph.add_node("router", router_node)
graph.add_node("tool", tool_node)
graph.add_node("synthesiser", synthesiser_node)

# Set the starting node
graph.set_entry_point("router")

# Connect nodes in order: router -> tool -> synthesiser -> END
graph.add_edge("router", "tool")
graph.add_edge("tool", "synthesiser")
graph.add_edge("synthesiser", END)

# Compile the graph into a runnable app
app = graph.compile()

print("LangGraph agent compiled successfully!")

LangGraph agent compiled successfully!


### Helper Function to Run the Agent

In [ ]:
def ask(question: str) -> AgentState:
    """Run a question through the agent and display the result."""

    print(f"\n{'='*65}")
    print(f" Question : {question}")
    print(f"{'='*65}")

    # Create initial AgentState
    initial_state: AgentState = {
        "question": question,
        "tool_used": "",
        "tool_output": "",
        "final_response": ""
    }

    # Run graph
    result = app.invoke(initial_state)

    print(f" Tool Used   : {result['tool_used']}")
    print(f"\n Final Answer :\n{result['final_response']}")
    print(f"{'='*65}")

    return result


print("Helper ask() function ready!")

Helper ask() function ready!


In [ ]:
ask("What is Retrieval-Augmented Generation?")


 Question : What is Retrieval-Augmented Generation?
[Router] Tool selected: arxiv
 Tool Used   : arxiv

 Final Answer :
Retrieval-Augmented Generation (RAG) is a type of natural language processing (NLP) technique that combines the strengths of retrieval-based and generation-based approaches. 

In traditional generation-based models, the system relies solely on its learned patterns and associations to produce text. In contrast, RAG models augment this process by retrieving relevant information from a large database or knowledge base, which is then used to inform and improve the generated text.

The retrieval component of RAG allows the model to access a vast amount of external knowledge, reducing the need for the model to memorize and store all possible information internally. This approach enables RAG models to generate more accurate, informative, and context-specific text, making them particularly useful for applications such as question-answering, text summarization, and conversati

{'question': 'What is Retrieval-Augmented Generation?',
 'tool_used': 'arxiv',
 'tool_output': 'Arxiv tool error: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=What+is+Retrieval-Augmented+Generation%3F&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=3)',
 'final_response': 'Retrieval-Augmented Generation (RAG) is a type of natural language processing (NLP) technique that combines the strengths of retrieval-based and generation-based approaches. \n\nIn traditional generation-based models, the system relies solely on its learned patterns and associations to produce text. In contrast, RAG models augment this process by retrieving relevant information from a large database or knowledge base, which is then used to inform and improve the generated text.\n\nThe retrieval component of RAG allows the model to access a vast amount of external knowledge, reducing the need for the model to memorize and store all possible information internall

In [ ]:
ask("What are the latest research papers on large language models?")


 Question : What are the latest research papers on large language models?
[Router] Tool selected: arxiv
 Tool Used   : arxiv

 Final Answer :
Unfortunately, I was unable to retrieve the latest research papers on large language models due to an error with the Arxiv tool. The page request resulted in an HTTP 429 error, which indicates that the request was blocked due to excessive usage.

However, I can suggest alternative ways to find the latest research papers on large language models:

1. **Visit the Arxiv website directly**: You can visit the Arxiv website (https://arxiv.org/) and search for papers related to large language models using relevant keywords such as "large language models," "transformers," or "natural language processing."
2. **Use academic search engines**: You can use academic search engines such as Google Scholar (https://scholar.google.com/) or Microsoft Academic (https://academic.microsoft.com/) to search for research papers on large language models.
3. **Check rese

{'question': 'What are the latest research papers on large language models?',
 'tool_used': 'arxiv',
 'tool_output': 'Arxiv tool error: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=What+are+the+latest+research+papers+on+large+language+models%3F&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=3)',
 'final_response': 'Unfortunately, I was unable to retrieve the latest research papers on large language models due to an error with the Arxiv tool. The page request resulted in an HTTP 429 error, which indicates that the request was blocked due to excessive usage.\n\nHowever, I can suggest alternative ways to find the latest research papers on large language models:\n\n1. **Visit the Arxiv website directly**: You can visit the Arxiv website (https://arxiv.org/) and search for papers related to large language models using relevant keywords such as "large language models," "transformers," or "natural language processing."\n2. **Use academ

In [ ]:
ask("Who invented the Transformer architecture?")


 Question : Who invented the Transformer architecture?
[Router] Tool selected: llm
 Tool Used   : llm

 Final Answer :
The Transformer architecture was introduced by Vaswani et al. in a research paper titled "Attention Is All You Need" in 2017. The authors, Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin, were researchers at Google at the time. They proposed the Transformer model as a replacement for traditional recurrent neural networks (RNNs) and convolutional neural networks (CNNs) for sequence-to-sequence tasks, such as machine translation. The Transformer architecture relies entirely on self-attention mechanisms, eliminating the need for recurrent connections and making it more parallelizable and efficient for large-scale computations.


{'question': 'Who invented the Transformer architecture?',
 'tool_used': 'llm',
 'tool_output': 'The Transformer architecture was introduced by Vaswani et al. in a research paper titled "Attention Is All You Need" in 2017. The authors, Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin, were researchers at Google at the time. They proposed the Transformer model as a replacement for traditional recurrent neural networks (RNNs) and convolutional neural networks (CNNs) for sequence-to-sequence tasks, such as machine translation. The Transformer architecture relies entirely on self-attention mechanisms, eliminating the need for recurrent connections and making it more parallelizable and efficient for large-scale computations.',
 'final_response': 'The Transformer architecture was introduced by Vaswani et al. in a research paper titled "Attention Is All You Need" in 2017. The authors, Ashish Vaswani, Noam Shazeer, Nik

In [ ]:
ask("What is the capital of Australia?")


 Question : What is the capital of Australia?
[Router] Tool selected: llm
 Tool Used   : llm

 Final Answer :
The capital of Australia is Canberra.


{'question': 'What is the capital of Australia?',
 'tool_used': 'llm',
 'tool_output': 'The capital of Australia is Canberra.',
 'final_response': 'The capital of Australia is Canberra.'}

In [ ]:
ask("What is the distance from visakhapatnam to bengaluru?")


 Question : What is the distance from visakhapatnam to bengaluru?
[Router] Tool selected: llm
 Tool Used   : llm

 Final Answer :
The distance from Visakhapatnam to Bengaluru is approximately 1,005 kilometers (624 miles). The travel time can vary depending on the mode of transportation and the route taken. 

Here are some approximate travel times and distances:
- By road: 1,005 km (624 miles), 15-18 hours
- By train: 760-970 km (472-603 miles), 12-15 hours (depending on the train route)
- By air: 760 km (472 miles), 1.5 hours (flight duration)


{'question': 'What is the distance from visakhapatnam to bengaluru?',
 'tool_used': 'llm',
 'tool_output': 'The distance from Visakhapatnam to Bengaluru is approximately 1,005 kilometers (624 miles). The travel time can vary depending on the mode of transportation and the route taken. \n\nHere are some approximate travel times and distances:\n- By road: 1,005 km (624 miles), 15-18 hours\n- By train: 760-970 km (472-603 miles), 12-15 hours (depending on the train route)\n- By air: 760 km (472 miles), 1.5 hours (flight duration)',
 'final_response': 'The distance from Visakhapatnam to Bengaluru is approximately 1,005 kilometers (624 miles). The travel time can vary depending on the mode of transportation and the route taken. \n\nHere are some approximate travel times and distances:\n- By road: 1,005 km (624 miles), 15-18 hours\n- By train: 760-970 km (472-603 miles), 12-15 hours (depending on the train route)\n- By air: 760 km (472 miles), 1.5 hours (flight duration)'}

In [ ]:
class MemoryAgentState(TypedDict):
    question: str         # Current question
    tool_used: str        # Tool selected
    tool_output: str      # Raw tool output
    final_response: str   # Final answer
    history: List[str]    # List of past Q&A turns

print("MemoryAgentState defined!")

MemoryAgentState defined!


In [ ]:
def memory_router_node(state: MemoryAgentState) -> MemoryAgentState:
    """Routes the question considering past conversation history."""
    question = state["question"]
    history = state["history"]

    # Show last 4 turns of history to LLM for context
    history_text = "\n".join(history[-4:]) if history else "No history yet."

    prompt = f"""You are a routing assistant. Use the conversation history and current question to pick the right tool.

Available tools:
- wikipedia  : for concepts, definitions, history, general knowledge
- arxiv      : for latest research papers and academic topics
- llm        : for comparisons, follow-up questions, opinions, simple facts
- out_of_scope: if the question is irrelevant or cannot be answered

Conversation History:
{history_text}

Current Question: {question}

Reply with ONLY one word from: wikipedia, arxiv, llm, out_of_scope"""

    response = llm.invoke(prompt)
    tool = response.content.strip().lower()

    valid_tools = ["wikipedia", "arxiv", "llm", "out_of_scope"]
    if tool not in valid_tools:
        tool = "llm"

    print(f"[Router] Tool selected: {tool}")
    return {**state, "tool_used": tool}

print("Memory router node defined!")

Memory router node defined!


In [ ]:
def memory_tool_node(state: MemoryAgentState) -> MemoryAgentState:
    """Runs the selected tool (same logic as before)."""
    question = state["question"]
    tool = state["tool_used"]

    if tool == "wikipedia":
        output = wikipedia_tool(question)
    elif tool == "arxiv":
        output = arxiv_tool(question)
    elif tool == "llm":
        output = llm_tool(question)
    elif tool == "out_of_scope":
        output = "Sorry, this question is out of scope for this research assistant."
    else:
        output = "Unknown tool."

    return {**state, "tool_output": output}

print("Memory tool node defined!")

Memory tool node defined!


In [ ]:
def memory_synthesiser_node(state: MemoryAgentState) -> MemoryAgentState:
    """Generates final answer using history + tool output, then saves to history."""
    question = state["question"]
    tool_output = state["tool_output"]
    tool = state["tool_used"]
    history = state["history"]

    history_text = "\n".join(history[-4:]) if history else "No history yet."

    # If out of scope or direct LLM answer, use as-is
    if tool in ["out_of_scope", "llm"]:
        final = tool_output
    else:
        prompt = f"""You are a research assistant in a conversation. Use the history and information below to give a clear answer.

Conversation History:
{history_text}

Current Question: {question}

Information:
{tool_output}

Give a clear, concise, well-structured answer:"""

        try:
            response = llm.invoke(prompt)
            final = response.content
        except Exception:
            final = tool_output

    # Append this Q&A pair to history
    updated_history = history + [
        f"User: {question}",
        f"Assistant: {final}"
    ]

    return {**state, "final_response": final, "history": updated_history}

print("Memory synthesiser node defined!")

Memory synthesiser node defined!


In [ ]:
memory_graph = StateGraph(MemoryAgentState)

memory_graph.add_node("router", memory_router_node)
memory_graph.add_node("tool", memory_tool_node)
memory_graph.add_node("synthesiser", memory_synthesiser_node)

memory_graph.set_entry_point("router")
memory_graph.add_edge("router", "tool")
memory_graph.add_edge("tool", "synthesiser")
memory_graph.add_edge("synthesiser", END)

memory_app = memory_graph.compile()

print("Memory agent compiled!")

Memory agent compiled!


### Chat Helper Function (keeps history alive between turns)

In [ ]:
# This holds the conversation state across all turns
conversation_state = {
    "question": "",
    "tool_used": "",
    "tool_output": "",
    "final_response": "",
    "history": []
}

def chat(question: str):
    """Send a message and keep conversation history between turns."""
    global conversation_state

    print(f"\n{'='*65}")
    print(f" You       : {question}")
    print(f"{'='*65}")

    # Update current question, keep the rest of state the same
    conversation_state["question"] = question

    result = memory_app.invoke(conversation_state)

    # Save full result (including updated history) back to state
    conversation_state = result

    print(f" Tool Used : {result['tool_used']}")
    print(f"\n Assistant : {result['final_response']}")
    print(f"{'='*65}")

print("chat() function ready! Start a conversation below.")

chat() function ready! Start a conversation below.


In [ ]:
chat("What is a Transformer?")


 You       : What is a Transformer?
[Router] Tool selected: wikipedia
 Tool Used : wikipedia

 Assistant : Although the Wikipedia search was unsuccessful, I can provide a general overview of what a Transformer is. 

A Transformer is a type of neural network architecture introduced in 2017 by Vaswani et al. in the paper "Attention Is All You Need." It's primarily used for natural language processing (NLP) tasks, such as machine translation, text classification, and language generation. The Transformer model relies on self-attention mechanisms to weigh the importance of different input elements relative to each other, allowing it to handle sequential data like text efficiently.

The Transformer architecture consists of an encoder and a decoder. The encoder takes in a sequence of tokens (e.g., words or characters) and generates a continuous representation of the input sequence. The decoder then uses this representation to generate the output sequence, one token at a time.

Transformers h

In [ ]:
chat("How is it different from an RNN?")


 You       : How is it different from an RNN?
[Router] Tool selected: llm
 Tool Used : llm

 Assistant : A Transformer is different from a Recurrent Neural Network (RNN) in several key ways:

1. **Architecture**: RNNs are designed to handle sequential data by maintaining an internal state that captures information from previous time steps. Transformers, on the other hand, use self-attention mechanisms to weigh the importance of different input elements relative to each other.
2. **Sequential processing**: RNNs process input sequences one step at a time, using the previous hidden state to inform the next step. Transformers process the entire input sequence simultaneously, allowing for parallelization and faster computation.
3. **Attention mechanism**: RNNs typically use a fixed-size context window to capture dependencies, whereas Transformers use self-attention to weigh the importance of all input elements relative to each other. This allows Transformers to capture long-range dependenc

In [ ]:
chat("Which one would you use for text generation and why?")


 You       : Which one would you use for text generation and why?
[Router] Tool selected: llm
 Tool Used : llm

 Assistant : **Text Generation Options:**

For text generation, I would recommend using a combination of Natural Language Processing (NLP) and Deep Learning techniques. Some popular options include:

1. **Language Models (LMs)**: These are statistical models that predict the next word in a sequence of words. Examples include:
	* **Transformers** (e.g., BERT, RoBERTa): These models use self-attention mechanisms to weigh the importance of different words in a sentence.
	* **Recurrent Neural Networks (RNNs)** (e.g., LSTM, GRU): These models use recurrent connections to capture sequential dependencies in text data.
2. **Generative Adversarial Networks (GANs)**: These models consist of a generator network that produces text and a discriminator network that evaluates the generated text.
3. **Sequence-to-Sequence (Seq2Seq) Models**: These models use an encoder-decoder architecture 

In [ ]:
def evaluate_response(question: str, response: str) -> str:
    """Use LLM to score a response on 3 criteria (each out of 5)."""

    prompt = f"""Evaluate the AI response below. Score each criterion from 1 to 5.

Criteria:
- Accuracy     : Is the information factually correct?
- Relevance    : Does it directly and fully answer the question?
- Groundedness : Is it based on real facts, not made up?

Question : {question}
Response : {response}

Reply in this exact format (3 lines only):
Accuracy: <score>/5 - <one sentence reason>
Relevance: <score>/5 - <one sentence reason>
Groundedness: <score>/5 - <one sentence reason>"""

    result = llm.invoke(prompt)
    return result.content

print("Evaluation function ready!")

Evaluation function ready!


In [ ]:
# Test questions to evaluate
eval_questions = [
    "What is Retrieval-Augmented Generation?",
    "Who invented the Transformer architecture?",
    "What is the capital of Australia?"
]

print("Running evaluation on 3 queries...\n")

for question in eval_questions:
    print(f"\n{'='*65}")
    print(f" Question : {question}")
    print(f"{'='*65}")

    # Get the agent's response
    result = app.invoke({
        "question": question,
        "tool_used": "",
        "tool_output": "",
        "final_response": ""
    })

    response_text = result["final_response"]

    print(f" Tool Used   : {result['tool_used']}")
    print(f" Response    : {response_text[:200]}...")
    print(f"\n--- Evaluation Scores ---")

    scores = evaluate_response(question, response_text)
    print(scores)
    print(f"{'='*65}")

Running evaluation on 3 queries...


 Question : What is Retrieval-Augmented Generation?
[Router] Tool selected: arxiv
 Tool Used   : arxiv
 Response    : Retrieval-Augmented Generation (RAG) is a type of natural language processing (NLP) technique that combines the strengths of retrieval-based and generation-based approaches. 

In traditional generatio...

--- Evaluation Scores ---
Accuracy: 5/5 - The information provided about Retrieval-Augmented Generation is factually correct and aligns with known concepts in natural language processing.
Relevance: 5/5 - The response directly and fully answers the question by providing a clear explanation of Retrieval-Augmented Generation.
Groundedness: 4/5 - The response is based on real facts and concepts, but the inability to retrieve more detailed information from external sources slightly limits its groundedness.

 Question : Who invented the Transformer architecture?
[Router] Tool selected: llm
 Tool Used   : llm
 Response    : The Transforme